# Fine-tune CodeBERT — LAMPS
Dataset từ Google Drive `NT230/data/d1/`, code từ GitHub.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, shutil, sys
DRIVE_D1  = '/content/drive/My Drive/NT230/data/d1'

!git clone --depth=1 https://github.com/khoilv2005/NT230.git /content/NT230
for f in ['run.py', 'model.py']:
    shutil.copy(f'/content/NT230/models/codebert-malware-detector/code/{f}', f'/content/{f}')
    print(f'✅ {f}')

In [ ]:
for f in ['train.jsonl', 'val.jsonl', 'test.jsonl']:
    shutil.copy(f'{DRIVE_D1}/{f}', f'/content/{f}')
    print(f'✅ {f}  ({sum(1 for _ in open("/content/"+f))} records)')

In [ ]:
!pip install -q transformers==4.40.0 torch accelerate tqdm

In [ ]:
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '❌ No GPU')

In [ ]:
!python /content/run.py \
  --output_dir=/content/saved_models \
  --model_type=roberta \
  --tokenizer_name=microsoft/codebert-base \
  --model_name_or_path=microsoft/codebert-base \
  --do_train --do_eval --do_test \
  --train_data_file=/content/train.jsonl \
  --eval_data_file=/content/val.jsonl \
  --test_data_file=/content/test.jsonl \
  --epoch 4 \
  --block_size 400 \
  --train_batch_size 32 \
  --eval_batch_size 64 \
  --learning_rate 2e-5 \
  --max_grad_norm 1.0 \
  --evaluate_during_training \
  --seed 123456

In [ ]:
ckpt = '/content/saved_models/checkpoint-best-acc/model.bin'
print(f'✅ {os.path.getsize(ckpt)/1e6:.0f} MB' if os.path.exists(ckpt) else '❌ Not found')
shutil.copytree('/content/saved_models', f'{DRIVE_D1}/saved_models', dirs_exist_ok=True)
print(f'✅ Saved to Drive: {DRIVE_D1}/saved_models/')